# APBS electrostatics: protein only

Amber-parameterized APBS electrostatic potential map for a single protein
(`protein.pdb`). The whole pipeline lives in this notebook.

Pipeline:

1. Verify protonation and fix the protein (PropKa + PDBFixer).
2. Parameterize the protein with AmberTools/ff19SB (`pdb4amber`, `tleap`,
   ParmEd PQR export).
3. Solve APBS on the protein PQR to produce `protein.dx`.

Unlike `examples/browndye/complex_pqr.ipynb` (which used PDB2PQR for its
protein-only substrate), every case in `examples/apbs/` is parameterized with
AmberTools so the PQR charges/radii are internally consistent across the
protein, ligand, and complex examples.

Activate the AmberTools environment before launching Jupyter:

```bash
conda activate ambertools
```

In [ ]:
import os
import shutil
from pathlib import Path

import parmed as pmd

from mdpp.prep import fix_pdb, run_propka, write_apbs_input

# Input (this directory).
EXAMPLE_DIR = Path.cwd()
PROTEIN_PDB = EXAMPLE_DIR / "protein.pdb"

# Workspace layout: each stage owns a folder under tmp/, with transient
# tool files kept in <stage>/intermediate/.
TMP_ROOT = EXAMPLE_DIR / "tmp"
PREP_DIR = TMP_ROOT / "prep"
PREP_INTERMEDIATE = PREP_DIR / "intermediate"
AMBERTOOLS_DIR = TMP_ROOT / "ambertools"
AMBERTOOLS_INTERMEDIATE = AMBERTOOLS_DIR / "intermediate"
APBS_DIR = TMP_ROOT / "apbs"
APBS_INTERMEDIATE = APBS_DIR / "intermediate"
for path in (
    PREP_DIR,
    PREP_INTERMEDIATE,
    AMBERTOOLS_DIR,
    AMBERTOOLS_INTERMEDIATE,
    APBS_DIR,
    APBS_INTERMEDIATE,
):
    path.mkdir(parents=True, exist_ok=True)

# Protonation / titration.
PH = 7.4

# AmberTools (protein parameterization).
PROTEIN_FF = "leaprc.protein.ff19SB"
PB_RADII = "mbondi3"
STRIP_PROTEIN_H = True  # let tleap re-add hydrogens per ff19SB

# APBS physics defaults live in mdpp.prep.apbs; override per-call below if a
# system needs non-standard values.

# Export every shell-facing knob so the %%bash cells inherit the same
# configuration without re-typing defaults.
os.environ.update({
    "AMBERTOOLS_DIR": str(AMBERTOOLS_DIR),
    "AMBERTOOLS_INTERMEDIATE": str(AMBERTOOLS_INTERMEDIATE),
    "APBS_DIR": str(APBS_DIR),
    "APBS_INTERMEDIATE": str(APBS_INTERMEDIATE),
    "PROTEIN_FF": PROTEIN_FF,
    "PB_RADII": PB_RADII,
    "STRIP_PROTEIN_H": "1" if STRIP_PROTEIN_H else "0",
    "PH": f"{PH}",
})

## Step 1: Verify protonation and fix the protein

Run PropKa as an explicit pKa/protonation-state check at the target pH, then
add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
protein_pdb = PROTEIN_PDB
# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = PREP_DIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = PREP_DIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Parameterize the protein with AmberTools

Run the AmberTools pipeline on the fixed protein:

1. `pdb4amber` cleans the protein PDB for tleap (`-y` strips hydrogens so
   tleap re-adds them per ff19SB).
2. `tleap` loads ff19SB and saves the prmtop/rst7 topology.
3. ParmEd exports the topology to PQR (charge + radius per atom).

Outputs live in `tmp/ambertools/`; transient files in its `intermediate/` folder.

In [ ]:
# Stage the fixed protein into the AmberTools intermediate folder.
shutil.copy(protein_fixed_pdb, AMBERTOOLS_INTERMEDIATE / "protein_fixed.pdb")

In [ ]:
%%bash
set -euo pipefail
cd "$AMBERTOOLS_INTERMEDIATE"

echo "=== 1. pdb4amber ==="
pdb4amber_args=(-i protein_fixed.pdb -o protein_amber.pdb -d --no-conect)
if [[ "$STRIP_PROTEIN_H" == "1" ]]; then
    pdb4amber_args+=(-y)
fi
pdb4amber "${pdb4amber_args[@]}"

echo "=== 2. tleap (ff19SB protein) ==="
cat >tleap.in <<EOF
source $PROTEIN_FF

protein = loadpdb protein_amber.pdb

set default PBRadii $PB_RADII
saveamberparm protein protein.prmtop protein.rst7
quit
EOF
tleap -f tleap.in

In [ ]:
# ParmEd: convert each topology/coord pair to PQR (charge + radius per atom).
for stem in ("protein",):
    parm = pmd.load_file(
        str(AMBERTOOLS_INTERMEDIATE / f"{stem}.prmtop"),
        xyz=str(AMBERTOOLS_INTERMEDIATE / f"{stem}.rst7"),
    )
    parm.save(str(AMBERTOOLS_INTERMEDIATE / f"{stem}.pqr"), overwrite=True)
    for suffix in ("prmtop", "rst7", "pqr"):
        shutil.copy(
            AMBERTOOLS_INTERMEDIATE / f"{stem}.{suffix}",
            AMBERTOOLS_DIR / f"{stem}.{suffix}",
        )

print("AmberTools PQR outputs:")
for stem in ("protein",):
    print("  ", AMBERTOOLS_DIR / f"{stem}.pqr")

## Step 3: Solve APBS for the protein

Generate an APBS input file (`protein.in`) for the protein PQR and run `apbs`
to produce the electrostatic potential map `protein.dx`. The grid is sized from
the radius-inflated atom bounding box of `protein.pqr` and padded by the
fine/coarse padding defaults in `mdpp.prep.apbs`.

In [ ]:
# Stage protein.pqr into the APBS intermediate folder and write protein.in.
shutil.copy(
    AMBERTOOLS_DIR / "protein.pqr",
    APBS_INTERMEDIATE / "protein.pqr",
)
apbs_in = write_apbs_input("protein", APBS_INTERMEDIATE)
print(f"APBS input -> {apbs_in}")

In [ ]:
%%bash
set -euo pipefail
cd "$APBS_INTERMEDIATE"

stem=protein
echo "=== APBS: $stem ==="
apbs "$stem.in" 2>&1 | tee "$stem.apbs.log"

if [[ -s "$stem-PE0.dx" ]]; then
    mv "$stem-PE0.dx" "$stem.dx"
elif [[ -s "$stem.pqr.dx" ]]; then
    mv "$stem.pqr.dx" "$stem.dx"
fi

if [[ ! -s "$stem.dx" ]]; then
    echo "Missing $stem.dx" >&2
    exit 1
fi
cp "$stem.in" "$APBS_DIR/$stem.in"
cp "$stem.apbs.log" "$APBS_DIR/$stem.apbs.log"
cp "$stem.dx" "$APBS_DIR/$stem.dx"
ls -lh "$APBS_DIR/$stem.dx"